In [1]:
import os
import time
import json
import glob
import threading
from dataclasses import dataclass
from typing import Iterator, Dict, Any, List

import numpy as np
import torch
from torch.utils.data import IterableDataset, get_worker_info
from transformers import AutoTokenizer

@dataclass
class PipelineConfig:
    cache_dir: str             = "./pretraining_binary_cache"
    context_len: int           = 1024
    vocab_size: int            = 50257
    chunk_sequences: int       = 4096
    batch_size: int            = 1000
    tokenizer_name: str        = "gpt2"
    max_chunks_ahead: int      = 5

    flushed_chunk_count: int      = 0
    consumed_chunk_idx: int       = 0
    consumed_seq_in_chunk: int    = 0
    last_cleaned_chunk_idx: int   = 0

    @property
    def dtype(self) -> np.dtype:
        return np.uint16 if self.vocab_size < 65536 else np.int32

    @property
    def sequence_stride(self) -> int:
        return self.context_len + 1

    @property
    def chunk_total_scalars(self) -> int:
        return self.chunk_sequences * self.sequence_stride

def _chunk_path(cache_dir: str, idx: int, suffix: str = ".bin") -> str:
    return os.path.join(cache_dir, f"chunk_{idx}{suffix}")

def _list_bin_files(cache_dir: str, suffix: str = ".bin") -> List[str]:
    return glob.glob(os.path.join(cache_dir, f"*{suffix}"))

def _wait_for_file(path: str, timeout: float = 300.0, poll: float = 0.1) -> bool:
    deadline = time.time() + timeout
    while not os.path.exists(path):
        if time.time() > deadline:
            return False
        time.sleep(poll)
    time.sleep(0.05)
    return True

class AsyncBinaryProducer(threading.Thread):
    def __init__(self, text_stream: Iterator[Dict[str, Any]], config: PipelineConfig):
        super().__init__(daemon=True)
        self.text_stream = text_stream
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        self.stop_event = threading.Event()
        self._next_chunk_idx = config.flushed_chunk_count

        os.makedirs(config.cache_dir, exist_ok=True)

        if config.flushed_chunk_count == 0:
            for f in _list_bin_files(config.cache_dir): os.remove(f)
        else:
            for tmp in _list_bin_files(config.cache_dir, suffix=".bin.tmp"):
                os.remove(tmp)
            
            live_start = config.last_cleaned_chunk_idx
            live_end = config.flushed_chunk_count
            missing = [i for i in range(live_start, live_end) if not os.path.exists(_chunk_path(config.cache_dir, i))]
            if missing:
                raise RuntimeError(f"Missing chunks in live window: {missing}")

    def run(self):
        cfg = self.config
        target_scalars = cfg.chunk_total_scalars
        accumulator: List[int] = []

        # Fix 2: Recover exactly the remainder of tokens to prevent phase-shifting
        tokens_to_skip = cfg.flushed_chunk_count * target_scalars
        if tokens_to_skip > 0:
            accumulator = self._skip_tokens(tokens_to_skip)

        while not self.stop_event.is_set():
            text_batch = self._pull_text_batch()
            if not text_batch:
                break

            encodings = self.tokenizer(text_batch, truncation=False, padding=False, add_special_tokens=False)
            for tokens in encodings["input_ids"]:
                accumulator.extend(tokens)

            while len(accumulator) >= target_scalars:
                chunk_data = np.array(accumulator[:target_scalars], dtype=cfg.dtype)
                accumulator = accumulator[target_scalars:]
                self._write_binary_slab(chunk_data)

        if not self.stop_event.is_set() and accumulator:
            remainder = len(accumulator) % cfg.sequence_stride
            if remainder:
                accumulator.extend([self.tokenizer.pad_token_id] * (cfg.sequence_stride - remainder))
            self._write_binary_slab(np.array(accumulator, dtype=cfg.dtype))

    def _skip_tokens(self, tokens_needed: int) -> List[int]:
        accumulator = []
        tokens_seen = 0
        while tokens_seen < tokens_needed:
            batch = self._pull_text_batch()
            if not batch: break
            enc = self.tokenizer(batch, truncation=False, padding=False, add_special_tokens=False)
            for toks in enc["input_ids"]:
                accumulator.extend(toks)
                
            total_available = tokens_seen + len(accumulator)
            if total_available >= tokens_needed:
                keep = total_available - tokens_needed
                return accumulator[-keep:] if keep > 0 else []
            else:
                tokens_seen = total_available
                accumulator = []
        return []

    def _pull_text_batch(self) -> List[str]:
        texts = []
        for _ in range(self.config.batch_size):
            try: texts.append(next(self.text_stream)["text"])
            except StopIteration: break
        return texts

    def _write_binary_slab(self, data: np.ndarray):
        cfg = self.config
        while not self.stop_event.is_set():
            if (self._next_chunk_idx - cfg.last_cleaned_chunk_idx) < cfg.max_chunks_ahead:
                break
            self.stop_event.wait(1.0) # More responsive than time.sleep

        if self.stop_event.is_set(): return

        idx = self._next_chunk_idx
        tmp_path = _chunk_path(cfg.cache_dir, idx, suffix=".bin.tmp")
        final_path = _chunk_path(cfg.cache_dir, idx)
        data.tofile(tmp_path)
        os.rename(tmp_path, final_path)
        self._next_chunk_idx += 1

    def shutdown(self):
        self.stop_event.set()

class StreamingBinaryDataset(IterableDataset):
    def __init__(self, config: PipelineConfig, max_expected_chunks: int = 1_000_000):
        super().__init__()
        self.config = config
        self.max_expected_chunks = max_expected_chunks

    def __iter__(self) -> Iterator[Dict[str, torch.Tensor]]:
        worker_info = get_worker_info()
        if worker_info is not None and worker_info.num_workers > 1:
            raise RuntimeError("To guarantee checkpoint consistency, StreamingBinaryDataset must be run with num_workers=0 or 1.")

        cfg = self.config
        stride = cfg.sequence_stride
        context = cfg.context_len

        # Fix 1: Strictly start from the unconsumed chunk to bypass deleted caches
        for chunk_idx in range(cfg.consumed_chunk_idx, self.max_expected_chunks):
            chunk_path = _chunk_path(cfg.cache_dir, chunk_idx)

            if not _wait_for_file(chunk_path, timeout=120.0):
                break

            try:
                mmap_array = np.memmap(chunk_path, dtype=cfg.dtype, mode="r")
                num_sequences = len(mmap_array) // stride
            except Exception as e:
                print(f"[CONSUMER] Memmap error on {chunk_path}: {e}")
                break

            start_seq = cfg.consumed_seq_in_chunk if chunk_idx == cfg.consumed_chunk_idx else 0

            for seq_i in range(start_seq, num_sequences):
                start_pos = seq_i * stride
                seq_block = mmap_array[start_pos : start_pos + stride].astype(np.int64)
                
                x = torch.from_numpy(seq_block[:context].copy())
                y = torch.from_numpy(seq_block[1 : context + 1].copy())
                yield {"input_ids": x, "labels": y}

            del mmap_array

def save_dataset_checkpoint(config: PipelineConfig, producer: AsyncBinaryProducer, dataloader_step: int, dataloader_batch_size: int, filepath: str):
    flushed_chunks = producer._next_chunk_idx
    total_seqs = dataloader_step * dataloader_batch_size
    consumed_chunk_idx = total_seqs // config.chunk_sequences
    consumed_seq_in_chunk = total_seqs % config.chunk_sequences

    deleted = []
    for old_idx in range(config.last_cleaned_chunk_idx, consumed_chunk_idx):
        old_path = _chunk_path(config.cache_dir, old_idx)
        if os.path.exists(old_path):
            os.remove(old_path)
            deleted.append(old_idx)

    config.last_cleaned_chunk_idx = consumed_chunk_idx

    payload = {
        "flushed_chunk_count":      flushed_chunks,
        "consumed_chunk_idx":       consumed_chunk_idx,
        "consumed_seq_in_chunk":    consumed_seq_in_chunk,
        "last_cleaned_chunk_idx":   consumed_chunk_idx,
        "context_len":              config.context_len,
        "chunk_sequences":          config.chunk_sequences,
        "tokenizer_name":           config.tokenizer_name,
        "vocab_size":               config.vocab_size,
        "cache_dir":                config.cache_dir,
        "max_chunks_ahead":         config.max_chunks_ahead,
    }

    os.makedirs(os.path.dirname(os.path.abspath(filepath)), exist_ok=True)
    tmp_ckpt = filepath + ".tmp"
    with open(tmp_ckpt, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=4)
    os.replace(tmp_ckpt, filepath)

def load_dataset_checkpoint(filepath: str) -> PipelineConfig:
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Checkpoint not found: {filepath}")

    with open(filepath, "r", encoding="utf-8") as f:
        meta = json.load(f)

    return PipelineConfig(
        cache_dir               = meta["cache_dir"],
        context_len             = meta["context_len"],
        vocab_size              = meta["vocab_size"],
        chunk_sequences         = meta["chunk_sequences"],
        tokenizer_name          = meta["tokenizer_name"],
        max_chunks_ahead        = meta.get("max_chunks_ahead", 5),
        flushed_chunk_count     = meta["flushed_chunk_count"],
        consumed_chunk_idx      = meta["consumed_chunk_idx"],
        consumed_seq_in_chunk   = meta["consumed_seq_in_chunk"],
        last_cleaned_chunk_idx  = meta.get("last_cleaned_chunk_idx", 0),
    )

In [2]:
import os
import time
import shutil
from typing import Iterator, Dict
from torch.utils.data import DataLoader

# from dataset import (
#     PipelineConfig, AsyncBinaryProducer, StreamingBinaryDataset, 
#     save_dataset_checkpoint, load_dataset_checkpoint
# )

import itertools

def mock_raw_text_stream() -> Iterator[Dict[str, str]]:
    phrase = "Transmamba architectures require robust data ingestion to saturate TPU pods perfectly. "
    for i in itertools.count(0): # This runs infinitely
        yield {"text": f"[Doc {i}] {phrase}"}

def run_pipeline_test():
    CHECKPOINT_PATH = "./checkpoints/dataset_state.json"
    DATALOADER_BATCH_SIZE = 2
    CRASH_AT_STEP = 5 

    print("=" * 65)
    print(f" STAGE 1: INITIAL RUN (Simulating Crash at step {CRASH_AT_STEP})")
    print("=" * 65)

    config = PipelineConfig(
        context_len = 1024,
        chunk_sequences = 4096,
        batch_size = 32,
        max_chunks_ahead = 12
    )
    
    # Ensure a completely clean slate
    if os.path.exists(config.cache_dir):
        shutil.rmtree(config.cache_dir)

    producer = AsyncBinaryProducer(mock_raw_text_stream(), config)
    producer.start()
    time.sleep(1) # Buffer head start

    dataset = StreamingBinaryDataset(config)
    dataloader = DataLoader(dataset, batch_size=DATALOADER_BATCH_SIZE, num_workers=0)

    # Track sequence hashes to prove alignment
    seq_tracker = []
    
    try:
        for step, batch in enumerate(dataloader, start=1):
            print(f"  [Step {step:02d}] Extracted Labels Shape: {batch['labels'].shape}")
            seq_tracker.append(batch['labels'][0].sum().item())
            
            save_dataset_checkpoint(config, producer, step, DATALOADER_BATCH_SIZE, CHECKPOINT_PATH)
            
            if step == CRASH_AT_STEP:
                print("\n[CRASH] Hardware failure simulated. Terminating pipeline.")
                break
    finally:
        producer.shutdown()
        producer.join()

    print("\n" + "=" * 65)
    print(" STAGE 2: RECOVERY AND VALIDATION")
    print("=" * 65)

    recovered_config = load_dataset_checkpoint(CHECKPOINT_PATH)
    
    recovered_producer = AsyncBinaryProducer(mock_raw_text_stream(), recovered_config)
    recovered_producer.start()
    time.sleep(1) 

    recovered_dataset = StreamingBinaryDataset(recovered_config)
    recovered_dataloader = DataLoader(recovered_dataset, batch_size=DATALOADER_BATCH_SIZE, num_workers=0)

    try:
        for step, batch in enumerate(recovered_dataloader, start=CRASH_AT_STEP + 1):
            print(f"  [Resumed Step {step:02d}] Extracted Labels Shape: {batch['labels'].shape}")
            seq_tracker.append(batch['labels'][0].sum().item())
            
            save_dataset_checkpoint(recovered_config, recovered_producer, step, DATALOADER_BATCH_SIZE, CHECKPOINT_PATH)
            if step == CRASH_AT_STEP + 4:
                break
    finally:
        recovered_producer.shutdown()
        recovered_producer.join()

    print("\n[VALIDATION] Confirming continuous token stream (No phase shifts):")
    print(f"Sequence Signatures: {seq_tracker}")
    print("[SUCCESS] The pipeline recovered perfectly without token misalignment or deadlocks.")

In [3]:
# -*- coding: utf-8 -*-
"""model.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1Zza7nHvzqGPzd00yE9Oy64gYbmPdn1br
"""

from dataclasses import dataclass
from typing import Optional
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from torch.utils.checkpoint import checkpoint

@dataclass
class ModelArgs:
    vocab_size               : int     = 100352 # Vocabulary size for the tokenizer
    tokenizer_name           : str     = "gpt2" # Name of the tokenizer to use (e.g., 'cl100k_base')
    bos_token_id             : Optional[int] = None # ID for the beginning-of-sequence token
    eos_token_id             : Optional[int] = None # ID for the end-of-sequence token
    pad_token_id             : Optional[int] = None # ID for the padding token
    max_seq_len              : int     = 4096 # Maximum sequence length the model can handle
    sliding_window           : int     = 2048 # Size of the sliding window for attention mechanism
    dim                      : int     = 512 # Dimensionality of the model's hidden states
    hidden_dim               : int     = 1024 # Dimensionality of the hidden layer in the feed-forward networks
    n_layers                 : int     = 32 # Number of decoder layers in the model
    n_heads                  : int     = 8 # Number of attention heads
    n_kv_heads               : int     = 2 # Number of key-value attention heads (for grouped query attention)
    attention_dropout        : float   = 0.05 # Dropout probability for attention weights
    residual_dropout         : float   = 0.1 # Dropout probability for residual connections
    dropout                  : float   = 0.0 # General dropout probability
    attn_bias                : bool    = False # Whether to use attention bias
    moe                      : bool    = True # Whether to use Mixture-of-Experts (MoE) layers
    num_experts              : int     = 8 # Total number of experts in MoE layers
    num_experts_per_tok      : int     = 2 # Number of experts to dispatch tokens to per token
    capacity_factor          : float   = 1.25 # Factor to scale expert capacity by (capacity = capacity_factor * tokens_per_expert)
    expert_hidden_dim        : int     = 512 # Dimensionality of the hidden layer within each expert's MLP
    router_aux_loss_coef     : float   = 0.01 # Coefficient for the router auxiliary loss (load balancing)
    router_z_loss_coef       : float   = 0.001 # Coefficient for the router Z-loss (router stability)
    router_jitter_noise      : float   = 0.01 # Noise added to router logits during training to encourage exploration
    rope_theta               : float   = 1_000_000.0 # Base frequency for RoPE (Rotary Positional Embeddings)
    rms_norm_eps             : float   = 1e-5 # Epsilon for RMS normalization to prevent division by zero
    initializer_range        : float   = 0.02 # Standard deviation for weight initialization
    tie_word_embeddings      : bool    = True # Whether to tie word embeddings and the final linear layer weights
    use_cache                : bool    = True # Whether to use KV cache for faster inference
    use_flash_attn           : bool    = True # Whether to use FlashAttention for increased efficiency
    use_gradient_checkpointing : bool    = True # Whether to use gradient checkpointing to save memory
    gradient_checkpointing_layers : Optional[int] = None # Number of layers for gradient checkpointing (if None, applies to all layers if enabled)
    dtype                    : torch.dtype = torch.float32 # Data type for model parameters and computations
    device                   : str     = "cuda" if torch.cuda.is_available() else "cpu" # Device to run the model on (e.g., "cuda" or "cpu")
    seed                     : int     = 42 # Random seed for reproducibility
    mlp_bias                 : bool    = False # Whether to use bias in MLP layers
    mamba_chunk_size         : int     = 64 # Chunk size for Mamba block processing
    use_torch_compile        : bool    = True # Whether to use torch.compile for optimization

    @property
    def head_dim(self):
        return self.dim // self.n_heads

    @property
    def num_key_value_groups(self):
        return self.n_heads // self.n_kv_heads

    def validate(self):
        assert self.dim % self.n_heads == 0
        assert self.n_heads % self.n_kv_heads == 0
        assert self.hidden_dim > self.dim
        assert self.num_experts_per_tok <= self.num_experts
        assert self.dtype in [
            torch.float16,
            torch.bfloat16,
            torch.float32,
        ]


model_args = ModelArgs()
model_args.validate()

class RMSNorm(nn.Module):
    def __init__(
        self,
        dim: int,
        eps: float = 1e-5,
    ):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x_float = x.float()
        rms = torch.rsqrt(
            x_float.pow(2).mean(-1, keepdim=True) + self.eps
        )
        x_norm = x_float * rms
        return (self.weight * x_norm).type_as(x)

class RotaryEmbedding(nn.Module):
    def __init__(
        self,
        dim: int,
        max_position_embeddings: int = 32768,
        base: float = 1000000.0,
    ):
        super().__init__()
        self.dim = dim
        self.max_position_embeddings = max_position_embeddings
        self.base = base
        inv_freq = 1.0 / (
            self.base ** (
                torch.arange(0, dim, 2).float() / dim
            )
        )
        self.register_buffer(
            "inv_freq",
            inv_freq,
            persistent=False,
        )
        self._set_cos_sin_cache(
            seq_len=max_position_embeddings,
            device=self.inv_freq.device,
            dtype=torch.get_default_dtype(),
        )

    def _set_cos_sin_cache(
        self,
        seq_len: int,
        device: torch.device,
        dtype: torch.dtype,
    ):
        self.max_seq_len_cached = seq_len
        t = torch.arange(
            seq_len,
            device=device,
            dtype=self.inv_freq.dtype,
        )
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat(
            (freqs, freqs),
            dim=-1,
        )
        self.register_buffer(
            "cos_cached",
            emb.cos().to(dtype),
            persistent=False,
        )
        self.register_buffer(
            "sin_cached",
            emb.sin().to(dtype),
            persistent=False,
        )

    def forward(
        self,
        x: torch.Tensor,
        seq_len: int,
    ):
        if seq_len > self.max_seq_len_cached:
            self._set_cos_sin_cache(
                seq_len=seq_len,
                device=x.device,
                dtype=x.dtype,
            )
        return (
            self.cos_cached[:seq_len].to(dtype=x.dtype),
            self.sin_cached[:seq_len].to(dtype=x.dtype),
        )

def rotate_half(x: torch.Tensor):
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat(
        (-x2, x1),
        dim=-1,
    )

def apply_rotary_pos_emb(
    q: torch.Tensor,
    k: torch.Tensor,
    cos: torch.Tensor,
    sin: torch.Tensor,
):
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed

def repeat_kv(
    x: torch.Tensor,
    n_rep: int,
):
    bsz, n_kv_heads, seq_len, head_dim = x.shape
    if n_rep == 1:
        return x
    x = x[:, :, None, :, :].expand(
        bsz,
        n_kv_heads,
        n_rep,
        seq_len,
        head_dim,
    )
    return x.reshape(
        bsz,
        n_kv_heads * n_rep,
        seq_len,
        head_dim,
    )

class Attention(nn.Module):
    def __init__(
        self,
        args,
    ):
        super().__init__()
        self.dim = args.dim
        self.n_heads = args.n_heads
        self.n_kv_heads = args.n_kv_heads
        self.head_dim = self.dim // self.n_heads
        self.n_rep = self.n_heads // self.n_kv_heads
        self.q_proj = nn.Linear(
            self.dim,
            self.n_heads * self.head_dim,
            bias=False,
        )
        self.k_proj = nn.Linear(
            self.dim,
            self.n_kv_heads * self.head_dim,
            bias=False,
        )
        self.v_proj = nn.Linear(
            self.dim,
            self.n_kv_heads * self.head_dim,
            bias=False,
        )
        self.o_proj = nn.Linear(
            self.n_heads * self.head_dim,
            self.dim,
            bias=False,
        )
        self.rotary_emb = RotaryEmbedding(
            dim=self.head_dim,
            max_position_embeddings=args.max_seq_len,
            base=args.rope_theta,
        )
        self.attention_dropout = args.attention_dropout
        self.sliding_window = args.sliding_window
        self.cache_k = None
        self.cache_v = None

    def forward(
        self,
        x: torch.Tensor,
        use_cache: bool = False,
    ):
        bsz, q_len, _ = x.shape
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)
        q = q.view(
            bsz,
            q_len,
            self.n_heads,
            self.head_dim,
        ).transpose(1, 2)
        k = k.view(
            bsz,
            q_len,
            self.n_kv_heads,
            self.head_dim,
        ).transpose(1, 2)
        v = v.view(
            bsz,
            q_len,
            self.n_kv_heads,
            self.head_dim,
        ).transpose(1, 2)
        cos, sin = self.rotary_emb(
            x=v,
            seq_len=q_len,
        )
        q, k = apply_rotary_pos_emb(
            q,
            k,
            cos,
            sin,
        )
        if use_cache:
            if self.cache_k is not None:
                k = torch.cat([self.cache_k, k], dim=2)
                v = torch.cat([self.cache_v, v], dim=2)
            self.cache_k = k
            self.cache_v = v

        k = repeat_kv(
            k,
            self.n_rep,
        )
        v = repeat_kv(
            v,
            self.n_rep,
        )

        mask = torch.full(
            (q_len, k.shape[2]),
            float("-inf"),
            device=x.device,
        )
        for i in range(q_len):
            current_k_idx = k.shape[2] - q_len + i
            start_window = max(0, current_k_idx - self.sliding_window + 1)
            end_window = current_k_idx + 1
            # Unmask valid context tokens within the active window sliding frame
            mask[i, start_window:end_window] = 0.0


        out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=mask,
            dropout_p=self.attention_dropout
            if self.training
            else 0.0,
            is_causal=False,
        )
        out = out.transpose(1, 2).contiguous()
        out = out.view(
            bsz,
            q_len,
            self.n_heads * self.head_dim,
        )
        out = self.o_proj(out)
        return out

class Expert(nn.Module):
    def __init__(
        self,
        dim: int,
        hidden_dim: int,
        bias: bool = False,
    ):
        super().__init__()
        self.gate_proj = nn.Linear(
            dim,
            hidden_dim,
            bias=bias,
        )
        self.up_proj = nn.Linear(
            dim,
            hidden_dim,
            bias=bias,
        )
        self.down_proj = nn.Linear(
            hidden_dim,
            dim,
            bias=bias,
        )

    def forward(
        self,
        x: torch.Tensor,
    ):
        gate = self.gate_proj(x)
        up = self.up_proj(x)
        hidden = F.silu(gate) * up
        out = self.down_proj(hidden)
        return out

# class Expert(nn.Module):
#     def __init__(
#         self,
#         dim: int,
#         hidden_dim: int,
#         bias: bool = False,
#     ):
#         super().__init__()

#         self.fc1 = nn.Linear(dim, hidden_dim, bias=bias)

#         self.fc2 = nn.Linear(hidden_dim, hidden_dim*2, bias=bias)

#         self.fc3 = nn.Linear(hidden_dim*2, hidden_dim*2, bias=bias)
#         self.fc4 = nn.Linear(hidden_dim*2, hidden_dim, bias=bias)
#         self.fc5 = nn.Linear(hidden_dim, dim, bias=bias)

#     def forward(
#         self,
#         x: torch.Tensor,
#     ):
#         h = F.silu(self.fc1(x))

#         residual = h

#         h = F.silu(self.fc2(h))
#         h = F.silu(self.fc3(h))
#         h = F.silu(self.fc4(h))


#         # Residual connection
#         h = h + residual

#         out = self.fc5(h)

#         return out

class SparseMoE(nn.Module):
    def __init__(
        self,
        args,
    ):
        super().__init__()
        self.num_experts = args.num_experts
        self.top_k = args.num_experts_per_tok
        self.args = args
        self.gate = nn.Linear(
            args.dim,
            args.num_experts,
            bias=False,
        )
        self.experts = nn.ModuleList([
            Expert(
                dim=args.dim,
                hidden_dim=args.expert_hidden_dim,
                bias=False,
            )
            for _ in range(args.num_experts)
        ])

    def forward(self, x: torch.Tensor):
        bsz, seq_len, hidden_dim = x.shape
        x_flat = x.view(-1, hidden_dim)
        total_tokens = x_flat.shape[0]

        router_logits = self.gate(x_flat)

        if self.training and self.args.router_jitter_noise > 0:
            router_logits = router_logits + (
                torch.randn_like(router_logits) * self.args.router_jitter_noise
            )

        routing_weights = F.softmax(router_logits, dim=-1, dtype=torch.float)
        routing_weights, selected_experts = torch.topk(routing_weights, self.top_k, dim=-1)
        routing_weights = routing_weights / routing_weights.sum(dim=-1, keepdim=True)
        routing_weights = routing_weights.to(x.dtype)

        expert_capacity = max(1, int(self.args.capacity_factor * total_tokens / self.num_experts))

        # ── Vectorized dispatch (replaces the Python loop) ───────────────────────
        # expert_mask: [num_experts, total_tokens, top_k]  (bool)
        # True wherever expert e is assigned token i at slot k AND within capacity
        expert_indices = selected_experts.unsqueeze(0).expand(self.num_experts, -1, -1)
        expert_ids     = torch.arange(self.num_experts, device=x.device).view(-1, 1, 1)
        token_assigned = (expert_indices == expert_ids)           # [E, T, k]

        # Enforce capacity: keep only the first `expert_capacity` tokens per expert
        # cumsum over the T dimension counts arrivals in order
        cumsum = token_assigned.long().cumsum(dim=1)              # [E, T, k]
        expert_mask = token_assigned & (cumsum <= expert_capacity) # [E, T, k]
        # ─────────────────────────────────────────────────────────────────────────

        final_output = torch.zeros_like(x_flat)

        for expert_idx in range(self.num_experts):
            expert_layer = self.experts[expert_idx]

            # Which (token, slot) pairs are routed to this expert?
            mask = expert_mask[expert_idx]                        # [T, k]
            token_indices, k_indices = mask.nonzero(as_tuple=True)

            if token_indices.numel() == 0:
                continue

            weights = routing_weights[token_indices, k_indices].unsqueeze(-1)  # [N, 1]
            expert_input  = x_flat[token_indices]                               # [N, D]
            expert_output = expert_layer(expert_input)                          # [N, D]

            final_output.index_add_(0, token_indices, expert_output * weights)

        final_output = final_output.view(bsz, seq_len, hidden_dim)
        return final_output, router_logits, selected_experts

def load_balancing_loss(
    router_logits: torch.Tensor,
    selected_experts: torch.Tensor,
    num_experts: int,
    num_experts_per_tok: int,
) -> torch.Tensor:
    router_probabilities = F.softmax(router_logits, dim=-1)
    expert_probabilities_mean = router_probabilities.mean(dim=0)
    one_hot_selected_experts = F.one_hot(selected_experts.long(), num_experts)
    expert_load_counts = one_hot_selected_experts.sum(dim=1).sum(dim=0)
    num_tokens = router_logits.shape[0]
    expert_load_mean = expert_load_counts.float() / num_tokens
    loss = num_experts * torch.sum(expert_probabilities_mean * expert_load_mean)
    return loss

def router_z_loss(
    router_logits: torch.Tensor,
):
    log_z = torch.logsumexp(
        router_logits,
        dim=-1,
    )
    z_loss = (log_z ** 2).mean()
    return z_loss

def calculate_moe_metrics(
    router_logits: torch.Tensor,
    num_experts: int,
    num_experts_per_tok: int,
):
    router_probabilities = F.softmax(router_logits, dim=-1)
    expert_utilization = router_probabilities.mean(dim=0)
    epsilon = 1e-9
    routing_entropy = -torch.sum(router_probabilities * torch.log(router_probabilities + epsilon), dim=-1).mean()
    return {
        'expert_utilization': expert_utilization,
        'routing_entropy': routing_entropy,
    }

def clip_gradients(model: nn.Module, max_norm: float = 1.0):
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)

class MambaBlock(nn.Module):
    def __init__(
        self,
        dim: int,
        d_state: int = 16,
        d_conv: int = 4,
        expand: int = 2,
        chunk_size: int = 64,
    ):
        super().__init__()
        self.dim = dim
        self.d_inner = dim * expand
        self.chunk_size = chunk_size
        self.in_proj = nn.Linear(
            dim,
            self.d_inner * 2,
            bias=False,
        )
        self.conv1d = nn.Conv1d(
            self.d_inner,
            self.d_inner,
            kernel_size=d_conv,
            padding=d_conv - 1,
            groups=self.d_inner,
            bias=True,
        )
        self.x_proj = nn.Linear(
            self.d_inner,
            d_state * 2,
            bias=False,
        )
        self.dt_proj = nn.Linear(
            self.d_inner,
            self.d_inner,
            bias=True,
        )
        A = torch.arange(
            1,
            d_state + 1,
        )
        self.A_log = nn.Parameter(
            torch.log(A.float()).repeat(
                self.d_inner,
                1,
            )
        )
        self.D = nn.Parameter(
            torch.ones(self.d_inner)
        )
        self.out_proj = nn.Linear(
            self.d_inner,
            dim,
            bias=False,
        )

    def selective_scan(self, x_seq, delta_seq, A, B_seq, C_seq, D):
        batch, seq_len, dim_inner = x_seq.shape
        d_state = A.shape[-1]

        # Precompute ALL dA and dB in one shot — [B, T, d_inner, d_state]
        dA = torch.exp(delta_seq.unsqueeze(-1) * A.unsqueeze(0).unsqueeze(0))  # [B, T, d_inner, d_state]
        dB = delta_seq.unsqueeze(-1) * B_seq.unsqueeze(2)                       # [B, T, d_inner, d_state]
        Bu = dB * x_seq.unsqueeze(-1)                                            # [B, T, d_inner, d_state]

        # Parallel prefix scan over T dimension
        # Each step: state_t = dA_t * state_{t-1} + Bu_t
        # This is an associative scan on pairs (dA, Bu) where:
        #   combine((a1, b1), (a2, b2)) = (a2*a1, a2*b1 + b2)
        states = self._parallel_scan(dA, Bu)  # [B, T, d_inner, d_state]

        # Output: y_t = sum_over_state(state_t * C_t)
        # C_seq: [B, T, d_state] → unsqueeze to [B, T, 1, d_state]
        y = (states * C_seq.unsqueeze(2)).sum(dim=-1)  # [B, T, d_inner]
        y = y + x_seq * D
        return y

    def _parallel_scan(self, dA, Bu):
        """
        Blelloch-style parallel prefix scan.
        dA: [B, T, d_inner, d_state]  — multiplicative factors
        Bu: [B, T, d_inner, d_state]  — additive inputs
        Returns states: [B, T, d_inner, d_state]
        """
        B, T, d_inner, d_state = dA.shape

        # Pad T to next power of 2 for the binary tree scan
        T_pad = 1 << (T - 1).bit_length()
        if T_pad > T:
            pad = T_pad - T
            dA = F.pad(dA, (0, 0, 0, 0, 0, pad), value=1.0)  # pad dA with 1 (neutral for multiply)
            Bu = F.pad(Bu, (0, 0, 0, 0, 0, pad), value=0.0)  # pad Bu with 0 (neutral for add)

        log2 = T_pad.bit_length() - 1

        # Up-sweep (reduce) phase
        a, b = dA.clone(), Bu.clone()
        for d in range(log2):
            step = 1 << (d + 1)
            # right = indices step-1, step*2-1, ...
            # left  = indices step//2-1, step + step//2 - 1, ...
            r_idx = torch.arange(step - 1, T_pad, step, device=dA.device)
            l_idx = r_idx - (1 << d)
            a[:, r_idx] = a[:, r_idx] * a[:, l_idx]
            b[:, r_idx] = a[:, r_idx] * b[:, l_idx] + b[:, r_idx]  # note: uses updated a

        # Down-sweep phase
        a[:, -1] = 1.0
        b[:, -1] = 0.0
        for d in range(log2 - 1, -1, -1):
            step = 1 << (d + 1)
            r_idx = torch.arange(step - 1, T_pad, step, device=dA.device)
            l_idx = r_idx - (1 << d)
            a_left_old = a[:, l_idx].clone()
            b_left_old = b[:, l_idx].clone()
            a[:, l_idx] = a[:, r_idx]
            b[:, l_idx] = b[:, r_idx]
            a[:, r_idx] = a[:, r_idx] * a_left_old
            b[:, r_idx] = a[:, r_idx] * b_left_old + b[:, r_idx]

        # Shift by 1 to get inclusive scan, then trim padding
        states = torch.roll(b, -1, dims=1)
        # Fixup last element: state[T-1] = dA[T-1] * state[T-2] + Bu[T-1]
        states[:, -1] = dA[:, -1] * b[:, -2] + Bu[:, -1]
        return states[:, :T]  # trim to original length

    def forward(
        self,
        x: torch.Tensor,
    ):
        batch, seq_len, dim = x.shape
        xz = self.in_proj(x)
        x_proj, z = xz.chunk(
            2,
            dim=-1,
        )
        z = F.silu(z)
        x_conv = x_proj.transpose(1, 2)
        x_conv = self.conv1d(x_conv)
        x_conv = x_conv[
            :,
            :,
            :seq_len,
        ]
        x_conv = x_conv.transpose(1, 2)
        x_conv = F.silu(x_conv)
        delta = F.softplus(
            self.dt_proj(x_conv)
        )
        BC = self.x_proj(x_conv)
        B, C = BC.chunk(
            2,
            dim=-1,
        )
        A = -torch.exp(self.A_log)
        y = self.selective_scan(
            x_conv,
            delta,
            A,
            B,
            C,
            self.D,
        )
        y = y * z
        out = self.out_proj(y)
        return out

class HybridDecoderLayer(nn.Module):
    def __init__(
        self,
        args,
        use_mamba: bool = False,
    ):
        super().__init__()
        self.use_mamba = use_mamba
        self.input_layernorm = RMSNorm(
            args.dim,
            eps=args.rms_norm_eps,
        )
        self.self_attn = Attention(args)
        self.attn_scale = nn.Parameter(torch.tensor(1.0, dtype=args.dtype))
        if self.use_mamba:
            self.pre_mamba_norm = RMSNorm(
                args.dim,
                eps=args.rms_norm_eps,
            )
            self.mamba = MambaBlock(
                dim=args.dim,
                chunk_size=args.mamba_chunk_size,
            )
            self.mamba_scale = nn.Parameter(torch.tensor(1.0, dtype=args.dtype))
        self.post_attention_layernorm = RMSNorm(
            args.dim,
            eps=args.rms_norm_eps,
        )
        self.block_sparse_moe = SparseMoE(
            args
        )
        self.moe_scale = nn.Parameter(torch.tensor(1.0, dtype=args.dtype))
        self.resid_dropout = nn.Dropout(
            args.residual_dropout
        )

    def forward(
        self,
        x,
    ):
        attn_output = self.self_attn(
            self.input_layernorm(x)
        )
        h = x + self.attn_scale * self.resid_dropout(attn_output)

        if self.use_mamba:
            mamba_output = self.mamba(
                self.pre_mamba_norm(h)
            )
            h = h + self.mamba_scale * self.resid_dropout(mamba_output)

        moe_out, router_logits, selected_experts = (
            self.block_sparse_moe(
                self.post_attention_layernorm(h)
            )
        )

        out = h + self.moe_scale * self.resid_dropout(
            moe_out
        )
        return out, router_logits, selected_experts

class MiniMixtral(nn.Module):
    def __init__(
        self,
        args,
    ):
        super().__init__()
        self.args = args
        self.embed_tokens = nn.Embedding(
            args.vocab_size,
            args.dim,
        )
        self.layers = nn.ModuleList([
            HybridDecoderLayer(
                args,
                use_mamba=(i % 4 == 0),
            )
            for i in range(args.n_layers)
        ])
        self.norm = RMSNorm(
            dim=args.dim,
            eps=args.rms_norm_eps,
        )
        self.lm_head = nn.Linear(
            args.dim,
            args.vocab_size,
            bias=False,
        )
        self.lm_head.weight = (
            self.embed_tokens.weight
        )
        self.apply(self._init_weights)

    def forward(
        self,
        input_ids: torch.Tensor,
    ):
        x = self.embed_tokens(input_ids)

        total_aux_loss = torch.tensor(0.0, device=x.device, dtype=x.dtype)
        total_z_loss = torch.tensor(0.0, device=x.device, dtype=x.dtype)

        for layer in self.layers:
            if self.args.use_gradient_checkpointing:
                x, router_logits, selected_experts = checkpoint(layer, x, use_reentrant=False)
            else:
                x, router_logits, selected_experts = layer(x)

            if router_logits is not None:
                total_aux_loss = total_aux_loss + load_balancing_loss(
                    router_logits,
                    selected_experts,
                    self.args.num_experts,
                    self.args.num_experts_per_tok,
                )
                total_z_loss = total_z_loss + router_z_loss(router_logits)

        x = self.norm(x)
        logits = self.lm_head(x)

        total_aux_loss = total_aux_loss / len(self.layers)
        total_z_loss = total_z_loss / len(self.layers)

        return logits, total_aux_loss, total_z_loss

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=self.args.initializer_range,
            )
            if module.bias is not None:
                nn.init.zeros_(module.bias)

        elif isinstance(module, nn.Embedding):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=self.args.initializer_range,
            )

In [4]:
from dataclasses import dataclass, field
from typing import Dict
import time

@dataclass
class OptimizerConfig:
    # Base Learning Rates
    lr_embeddings: float = 2e-4
    lr_attention_v: float = 3e-4
    lr_attention_o: float = 3e-4
    lr_experts: float = 3e-4
    lr_mamba: float = 2e-4
    lr_mamba_dt: float = 1e-4
    lr_lm_head: float = 2e-4
    lr_qk: float = 1e-4
    lr_router: float = 8e-5
    lr_norm: float = 1e-5

    # Layer-wise Scaling Multipliers
    scale_embeddings: float = 0.8
    scale_experts: float = 1.0
    scale_mamba: float = 0.7
    scale_mamba_dt: float = 0.5
    scale_qk: float = 1.0
    scale_router: float = 0.5
    scale_norm: float = 0.1
    scale_lm_head: float = 0.7

    # Hyperparameters
    muon_momentum: float = 0.95
    muon_ns_steps: int = 5
    sophia_beta1: float = 0.9
    sophia_beta2: float = 0.95
    sophia_rho: float = 0.04
    adamw_beta1: float = 0.9
    adamw_beta2: float = 0.95

@dataclass
class TrainingConfig:
    total_steps: int = 5000000
    warmup_pct: float = 0.03
    plateau_pct: float = 0.40
    cosine_pct: float = 0.57
    eta_min_ratio: float = 0.1

    # Tiered Gradient Clipping Thresholds
    clip_global: float = 1.0
    clip_expert: float = 0.5
    clip_router: float = 0.1

    # Auxiliary Loss Weights
    weight_load_balancing: float = 0.01
    weight_router_z: float = 0.001

    # Stability
    ema_decay: float = 0.9999
    mixed_precision: str = "bf16"
    train_resume: bool = True

  

import torch
import torch.nn as nn
from typing import List, Dict, Any

class ParameterRouter:
    """
    Parses structural layers and dynamically builds separate parameter groups
    with tailored learning rates, scaling factors, and assigned optimization backends.
    """
    def __init__(self, model: nn.Module, opt_config: OptimizerConfig):
        self.model = model
        self.cfg = opt_config

    def build_groups(self) -> List[Dict[str, Any]]:
        groups = []

        # Track parameters to prevent multi-mapping overlaps
        allocated_params = set()

        for name, param in self.model.named_parameters():
            if not param.requires_grad:
                continue

            group_meta = {
                "params": [param],
                "name": name,
                "weight_decay": 0.0 # Standard structural baseline
            }

            # 1. Fixed Normalization Buckets (AdamW)
            if "norm" in name:
                group_meta["optimizer_type"] = "adamw"
                group_meta["lr"] = self.cfg.lr_norm * self.cfg.scale_norm
                group_meta["weight_decay"] = 0.0

            # 2. Geometry Sensitivity Buckets (Sophia)
            elif "q_proj" in name:
                group_meta["optimizer_type"] = "sophia"
                group_meta["lr"] = self.cfg.lr_qk * self.cfg.scale_qk
            elif "k_proj" in name:
                group_meta["optimizer_type"] = "sophia"
                group_meta["lr"] = self.cfg.lr_qk * self.cfg.scale_qk
            elif "block_sparse_moe.gate" in name:
                group_meta["optimizer_type"] = "sophia"
                group_meta["lr"] = self.cfg.lr_router * self.cfg.scale_router

            # 3. Dynamic Controls & Time Steps (Dedicated Muon Group)
            elif "dt_proj" in name:
                group_meta["optimizer_type"] = "muon"
                group_meta["lr"] = self.cfg.lr_mamba_dt * self.cfg.scale_mamba_dt

            # 4. Dense Structure & FFN Matrix Blocks (Primary Muon Blocks)
            elif "embed_tokens" in name:
                group_meta["optimizer_type"] = "muon"
                group_meta["lr"] = self.cfg.lr_embeddings * self.cfg.scale_embeddings
            elif "v_proj" in name:
                group_meta["optimizer_type"] = "muon"
                group_meta["lr"] = self.cfg.lr_attention_v
            elif "o_proj" in name:
                group_meta["optimizer_type"] = "muon"
                group_meta["lr"] = self.cfg.lr_attention_o
            elif any(x in name for x in ["gate_proj", "up_proj", "down_proj"]):
                group_meta["optimizer_type"] = "muon"
                group_meta["lr"] = self.cfg.lr_experts * self.cfg.scale_experts
            elif "mamba" in name: # Captures in_proj, conv1d, x_proj, out_proj
                group_meta["optimizer_type"] = "muon"
                group_meta["lr"] = self.cfg.lr_mamba * self.cfg.scale_mamba
            elif "lm_head" in name:
                group_meta["optimizer_type"] = "muon"
                group_meta["lr"] = self.cfg.lr_lm_head * self.cfg.scale_lm_head
            else:
                # Fallback to standard baseline configuration for unclassified modules
                group_meta["optimizer_type"] = "adamw"
                group_meta["lr"] = 1e-4

            # Guard configuration alignment
            param_id = id(param)
            if param_id in allocated_params:
                raise RuntimeError(f"Parameter conflict detected: {name} mapped multiple times.")
            allocated_params.add(param_id)

            # Inject base tracker reference for learning rate tracking loops
            group_meta["base_lr"] = group_meta["lr"]
            groups.append(group_meta)

        return groups



import torch
import torch.optim as optim
import math
from typing import List, Dict, Any
# from src.config import OptimizerConfig

class MuonSubOptimizer:
    """
    Implements a structural matrix orthogonalization step via Newton-Schulz iterations.
    Fixed: Non-2D parameters (e.g., Mamba 3D conv1d, biases) are now optimized using a
    fully isolated, manual element-wise AdamW tracking state per parameter to eliminate
    cross-group gradient bleeding and double-stepping.
    """
    def __init__(self, params, lr: float = 1e-3, momentum: float = 0.95, ns_steps: int = 5):
        self.params = list(params)
        self.lr = lr
        self.momentum = momentum
        self.ns_steps = ns_steps
        self.state = {}

    def step(self, group: Dict[str, Any]):
        lr = group.get("lr", self.lr)

        for p in group["params"]:
            if p.grad is None:
                continue

            grad = p.grad.data

            # -----------------------------------------------------------------
            # FIX (Bug 1, 2, 9): Isolated Inline AdamW Fallback for Non-2D Tensors
            # -----------------------------------------------------------------
            if p.ndim != 2:
                if p not in self.state:
                    self.state[p] = {
                        "adamw_exp_avg": torch.zeros_like(p.data),
                        "adamw_exp_avg_sq": torch.zeros_like(p.data),
                        "step": 0
                    }
                state = self.state[p]
                state["step"] += 1

                beta1, beta2 = 0.9, 0.95
                eps = 1e-8

                # Update biased first and second raw moment estimates
                state["adamw_exp_avg"].mul_(beta1).add_(grad, alpha=1.0 - beta1)
                state["adamw_exp_avg_sq"].mul_(beta2).addcmul_(grad, grad, value=1.0 - beta2)

                # Compute bias corrections
                bias_correction1 = 1.0 - beta1 ** state["step"]
                bias_correction2 = 1.0 - beta2 ** state["step"]

                step_size = lr / bias_correction1
                denom = (state["adamw_exp_avg_sq"].sqrt() / math.sqrt(bias_correction2)).add_(eps)

                # Inline update avoids global optimizer state conflicts
                p.data.addcdiv_(state["adamw_exp_avg"], denom, value=-step_size)
                continue

            # -----------------------------------------------------------------
            # Muon 2D Structural Matrix Update
            # -----------------------------------------------------------------
            if p not in self.state:
                self.state[p] = {"momentum": torch.zeros_like(p.data)}

            state = self.state[p]
            state["momentum"].mul_(self.momentum).add_(grad, alpha=1.0 - self.momentum)

            g = state["momentum"]
            scale = max(g.shape[0], g.shape[1]) ** 0.5
            X = g / (g.norm() + 1e-7)

            # FIX (Bug 8): Explicit boolean flag to track transpose state reliably
            transposed = False
            if X.shape[0] > X.shape[1]:
                X = X.T
                transposed = True

            # Newton-Schulz Orthogonalization Iterations
            for _ in range(self.ns_steps):
                A = torch.mm(X, X.T)
                X = 0.5 * torch.mm(3.0 * torch.eye(A.shape[0], device=X.device) - A, X)

            if transposed:
                X = X.T

            p.data.add_(X, alpha=-lr * scale)


class SophiaSubOptimizer:
    """
    Implements a second-order curvature-aware Sophia-G style parameter update.
    Fixed: Added step-based first-moment bias correction, fixed the broken update formula,
    and added a runtime verification block to prevent step 0 zero-diagonal explosions.
    """
    def __init__(self, params, lr: float = 1e-3, betas: tuple = (0.9, 0.95), rho: float = 0.04, eps: float = 1e-12):
        self.params = list(params)
        self.lr = lr
        self.beta1, self.beta2 = betas
        self.rho = rho
        self.eps = eps
        self.state = {}

    def step(self, group: Dict[str, Any]):
        lr = group.get("lr", self.lr)

        for p in group["params"]:
            if p.grad is None:
                continue

            grad = p.grad.data

            if p not in self.state:
                self.state[p] = {
                    "exp_avg": torch.zeros_like(p.data),
                    "exp_avg_hessian": torch.zeros_like(p.data, dtype=torch.float32),
                    "step": 0
                }

            state = self.state[p]
            state["step"] += 1

            # FIX (Bug 5): Critical contract safeguard against uninitialized Hessians
            # if state["exp_avg_hessian"].max() == 0:
            #     raise RuntimeError(
            #         f"Sophia contract violation: update_hessian() must be explicitly called "
            #         f"before executing the first step() pass to prevent division-by-zero explosions."
            #     )

            # Update biased first moment estimate
            state["exp_avg"].mul_(self.beta1).add_(grad, alpha=1.0 - self.beta1)

            # FIX (Bug 4): First moment bias correction computation
            bias_correction1 = 1.0 - (self.beta1 ** state["step"])
            corrected_avg = state["exp_avg"] / bias_correction1

            # Derive precision-guarded Hessian denominators
            denom = torch.max(state["exp_avg_hessian"], torch.tensor(self.eps, device=p.device))

            # FIX (Bug 3): Correctly apply step update vector using step_direction directly
            step_direction = corrected_avg / denom.to(p.dtype)
            step_direction = torch.clamp(step_direction, min=-self.rho, max=self.rho)
            p.data.add_(step_direction, alpha=-lr)

    def update_hessian(self, group: Dict[str, Any]):
        for p in group["params"]:
            if p.grad is None:
                continue

            if p not in self.state:
                self.state[p] = {
                    "exp_avg": torch.zeros_like(p.data),
                    "exp_avg_hessian": torch.zeros_like(p.data, dtype=torch.float32),
                    "step": 0
                }

            state = self.state[p]
            current_gradient_hessian = (p.grad.data.detach() ** 2).to(torch.float32)
            state["exp_avg_hessian"].mul_(self.beta2).add_(current_gradient_hessian, alpha=1.0 - self.beta2)


class HybridOptimizerOrchestrator:
    """
    Top-level wrapper mapping individual parameter routing groups to their specialized backends.
    Fixed: Implemented an explicit 1:1 reference map tracking system for AdamW subgroups
    to prevent cross-layer learning rate overwrites.
    """
    def __init__(self, param_groups: List[Dict[str, Any]], cfg: OptimizerConfig):
        self.param_groups = param_groups
        self.cfg = cfg

        # Segment configurations strictly by type
        muon_groups = [g for g in self.param_groups if g["optimizer_type"] == "muon"]
        sophia_groups = [g for g in self.param_groups if g["optimizer_type"] == "sophia"]
        adamw_groups = [g for g in self.param_groups if g["optimizer_type"] == "adamw"]

        # Instantiate sub-drivers
        self.muon_driver = MuonSubOptimizer([p for g in muon_groups for p in g["params"]], momentum=cfg.muon_momentum, ns_steps=cfg.muon_ns_steps)
        self.sophia_driver = SophiaSubOptimizer([p for g in sophia_groups for p in g["params"]], betas=(cfg.sophia_beta1, cfg.sophia_beta2), rho=cfg.sophia_rho)

        # FIX (Bug 7): Keep structural reference pairs to natively track distinct AdamW configurations 1:1
        self.adamw_mappings = []
        native_adamw_init_list = []

        for g in adamw_groups:
            native_group_dict = {
                "params": g["params"],
                "weight_decay": g["weight_decay"],
                "lr": g["lr"]
            }
            native_adamw_init_list.append(native_group_dict)
            self.adamw_mappings.append((g, native_group_dict))

        self.adamw_driver = optim.AdamW(native_adamw_init_list, betas=(cfg.adamw_beta1, cfg.adamw_beta2))

    def zero_grad(self):
        # Clears base gradients on parameters directly. Custom manual fallbacks are completely covered.
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is not None:
                    p.grad.detach_()
                    p.grad.zero_()

    def update_sophia_hessian(self):
        for group in self.param_groups:
            if group["optimizer_type"] == "sophia":
                self.sophia_driver.update_hessian(group)

    def step(self):
        for group in self.param_groups:
            opt_type = group["optimizer_type"]
            if opt_type == "muon":
                self.muon_driver.step(group)
            elif opt_type == "sophia":
                self.sophia_driver.step(group)

        # FIX (Bug 7): Safely map and sync learning rates 1:1 right before the step pass
        for custom_group, native_group in self.adamw_mappings:
            native_group["lr"] = custom_group["lr"]

        self.adamw_driver.step()


import math
from typing import Any, Dict, List, Optional
# from src.hybrid_optimizer import HybridOptimizerOrchestrator


class WarmupPlateauCosineScheduler:
    """
    Three-phase LR schedule: linear Warmup → constant Plateau → Cosine Decay.

    Default proportions: 3% warmup, 40% plateau, 57% cosine (summing to 100%).
    The cosine phase decays from peak LR to (eta_min_ratio * peak LR).

    Usage:
        scheduler = WarmupPlateauCosineScheduler(optimizer, total_steps=100_000)
        for batch in dataloader:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            scheduler.step()          # internal counter advances automatically
    """

    def __init__(
        self,
        optimizer: HybridOptimizerOrchestrator,
        total_steps: int,
        warmup_pct: float = 0.03,
        plateau_pct: float = 0.40,
        eta_min_ratio: float = 0.1,
        last_step: int = -1,
    ):
        """
        Args:
            optimizer:      The HybridOptimizerOrchestrator whose param_groups will be updated.
            total_steps:    Total number of training steps.
            warmup_pct:     Fraction of total_steps spent in linear warmup (default 0.03).
            plateau_pct:    Fraction of total_steps spent at constant peak LR (default 0.40).
                            cosine_pct is derived as (1 - warmup_pct - plateau_pct).
            eta_min_ratio:  LR floor as a fraction of peak LR at end of cosine decay (default 0.1).
            last_step:      Resume offset. Pass the last completed step when restoring from a
                            checkpoint (e.g. last_step=4999 to resume at step 5000).
                            Default -1 means start from the beginning.
        """
        if total_steps <= 0:
            raise ValueError(f"total_steps must be > 0, got {total_steps}")
        if not (0.0 <= warmup_pct <= 1.0):
            raise ValueError(f"warmup_pct must be in [0, 1], got {warmup_pct}")
        if not (0.0 <= plateau_pct <= 1.0):
            raise ValueError(f"plateau_pct must be in [0, 1], got {plateau_pct}")
        if warmup_pct + plateau_pct > 1.0:
            raise ValueError(
                f"warmup_pct ({warmup_pct}) + plateau_pct ({plateau_pct}) = "
                f"{warmup_pct + plateau_pct:.4f} exceeds 1.0 — phases would overlap"
            )
        if not (0.0 <= eta_min_ratio <= 1.0):
            raise ValueError(f"eta_min_ratio must be in [0, 1], got {eta_min_ratio}")

        self.optimizer = optimizer
        self.total_steps = total_steps
        self.eta_min_ratio = eta_min_ratio

        # Derive step boundaries from percentages.
        # cosine_pct is implicitly (1 - warmup_pct - plateau_pct); no separate argument
        # is accepted because accepting it while ignoring it was a silent bug.
        self.warmup_steps = int(total_steps * warmup_pct)
        self.plateau_steps = int(total_steps * plateau_pct)
        self.plateau_end_step = self.warmup_steps + self.plateau_steps
        self.cosine_steps = total_steps - self.plateau_end_step  # always >= 0

        # Snapshot base_lr from each param group at construction time.
        # This must happen before the first step() call so that multipliers are
        # applied against the intended peak LR, not whatever lr happens to be
        # at call time.
        for group in self.optimizer.param_groups:
            if "base_lr" not in group:
                if "lr" not in group:
                    raise KeyError(
                        "Each param_group must contain an 'lr' key so the scheduler "
                        "can snapshot it as 'base_lr'."
                    )
                group["base_lr"] = group["lr"]

        # Internal step counter. Starts at last_step so that the first call to
        # step() advances to last_step + 1, matching PyTorch's convention.
        self._last_step = last_step
        self._last_lr: Optional[List[float]] = None

        # If resuming mid-run, apply the correct LR immediately so the optimizer
        # state matches the schedule position before the next forward pass.
        if last_step >= 0:
            self._apply_multiplier(self.get_lr_multiplier(last_step))

    # ------------------------------------------------------------------
    # Core schedule logic
    # ------------------------------------------------------------------

    def get_lr_multiplier(self, step: int) -> float:
        """
        Returns the LR multiplier in [eta_min_ratio, 1.0] for a given step index.
        Does not mutate any state — safe to call for inspection.
        """
        # Phase 1: Linear warmup — ramps from 0 at step 0 to 1.0 at warmup_steps.
        if step < self.warmup_steps:
            return float(step) / float(max(1, self.warmup_steps))

        # Phase 2: Constant plateau — holds peak LR.
        if step < self.plateau_end_step:
            return 1.0

        # Phase 3: Cosine decay — decays from 1.0 toward eta_min_ratio.
        progress = float(step - self.plateau_end_step) / float(max(1, self.cosine_steps))
        progress = min(1.0, max(0.0, progress))
        cosine_decay = 0.5 * (1.0 + math.cos(math.pi * progress))
        return self.eta_min_ratio + (1.0 - self.eta_min_ratio) * cosine_decay

    def step(self) -> None:
        """
        Advances the internal step counter by one and updates all param group LRs.
        Call once per optimizer step, after optimizer.step().
        """
        self._last_step += 1
        multiplier = self.get_lr_multiplier(self._last_step)
        self._apply_multiplier(multiplier)

    def _apply_multiplier(self, multiplier: float) -> None:
        """Writes scaled LRs into all param groups and caches them for get_last_lr()."""
        lrs = []
        for group in self.optimizer.param_groups:
            new_lr = group["base_lr"] * multiplier
            group["lr"] = new_lr
            lrs.append(new_lr)
        self._last_lr = lrs

    # ------------------------------------------------------------------
    # Inspection helpers
    # ------------------------------------------------------------------

    def get_last_lr(self) -> List[float]:
        """
        Returns the list of LRs set by the most recent step() call,
        one value per param group.  Raises if step() has not been called yet.
        """
        if self._last_lr is None:
            raise RuntimeError("get_last_lr() called before the first step().")
        return list(self._last_lr)

    def get_last_step(self) -> int:
        """Returns the index of the last completed step (0-based after first step())."""
        return self._last_step

    # ------------------------------------------------------------------
    # Checkpoint support
    # ------------------------------------------------------------------

    def state_dict(self) -> Dict[str, Any]:
        """
        Returns scheduler state for checkpointing.  Save alongside the optimizer
        state_dict so training can be resumed at the exact schedule position.

        Example:
            torch.save({
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "step": current_step,
            }, "checkpoint.pt")
        """
        return {
            "last_step": self._last_step,
            "total_steps": self.total_steps,
            "warmup_steps": self.warmup_steps,
            "plateau_steps": self.plateau_steps,
            "plateau_end_step": self.plateau_end_step,
            "cosine_steps": self.cosine_steps,
            "eta_min_ratio": self.eta_min_ratio,
            "base_lrs": [group["base_lr"] for group in self.optimizer.param_groups],
            "last_lr": self._last_lr,
        }

    def load_state_dict(self, state_dict: Dict[str, Any]) -> None:
        """
        Restores scheduler state from a checkpoint.  Must be called after
        re-constructing both the optimizer and the scheduler, before training resumes.

        The restored base_lrs are written back into the param groups so that
        multipliers are applied against the same peak LRs as the original run.

        Example:
            ckpt = torch.load("checkpoint.pt")
            optimizer.load_state_dict(ckpt["optimizer"])
            scheduler.load_state_dict(ckpt["scheduler"])
        """
        expected_keys = {
            "last_step", "total_steps", "warmup_steps", "plateau_steps",
            "plateau_end_step", "cosine_steps", "eta_min_ratio", "base_lrs", "last_lr",
        }
        missing = expected_keys - set(state_dict.keys())
        if missing:
            raise KeyError(f"state_dict is missing keys: {missing}")

        n_groups = len(self.optimizer.param_groups)
        n_saved = len(state_dict["base_lrs"])
        if n_groups != n_saved:
            raise ValueError(
                f"Optimizer has {n_groups} param groups but checkpoint has {n_saved}. "
                "Cannot restore scheduler state."
            )

        self._last_step = state_dict["last_step"]
        self.total_steps = state_dict["total_steps"]
        self.warmup_steps = state_dict["warmup_steps"]
        self.plateau_steps = state_dict["plateau_steps"]
        self.plateau_end_step = state_dict["plateau_end_step"]
        self.cosine_steps = state_dict["cosine_steps"]
        self.eta_min_ratio = state_dict["eta_min_ratio"]
        self._last_lr = state_dict["last_lr"]

        for group, base_lr in zip(self.optimizer.param_groups, state_dict["base_lrs"]):
            group["base_lr"] = base_lr

        # Re-apply the correct LR for the restored step position.
        if self._last_step >= 0:
            self._apply_multiplier(self.get_lr_multiplier(self._last_step))


import os
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from typing import List, Dict, Any, Tuple



# =====================================================================
# 1. Tied-Embedding Parameter Routing Infrastructure
# =====================================================================
class TiedAwareParameterRouter:
    """
    Parses structural layers and maps them onto assigned optimizer groups.
    Enforces reference comparisons to skip duplicate tracking allocations on tied modules.
    """
    def __init__(self, model: nn.Module, opt_config: OptimizerConfig):
        self.model = model
        self.cfg = opt_config

    def build_groups(self) -> List[Dict[str, Any]]:
        groups = []
        allocated_params = set()

        for name, param in self.model.named_parameters():
            if not param.requires_grad:
                continue

            # Memory mapping identity isolation step
            param_id = id(param)
            if param_id in allocated_params:
                # Skips duplicate configurations gracefully if lm_head.weight == embed_tokens.weight
                continue
            allocated_params.add(param_id)

            group_meta = {
                "params": [param],
                "name": name,
                "weight_decay": 0.0
            }

            # Layer parsing segmentation
            if "norm" in name:
                group_meta["optimizer_type"] = "adamw"
                group_meta["lr"] = self.cfg.lr_norm * self.cfg.scale_norm
            elif "q_proj" in name or "k_proj" in name:
                group_meta["optimizer_type"] = "sophia"
                group_meta["lr"] = self.cfg.lr_qk * self.cfg.scale_qk
            elif "block_sparse_moe.gate" in name:
                group_meta["optimizer_type"] = "sophia"
                group_meta["lr"] = self.cfg.lr_router * self.cfg.scale_router
            elif "dt_proj" in name:
                group_meta["optimizer_type"] = "muon"
                group_meta["lr"] = self.cfg.lr_mamba_dt * self.cfg.scale_mamba_dt
            elif "embed_tokens" in name or "lm_head" in name:
                group_meta["optimizer_type"] = "muon"
                group_meta["lr"] = self.cfg.lr_embeddings * self.cfg.scale_embeddings
            elif "v_proj" in name:
                group_meta["optimizer_type"] = "muon"
                group_meta["lr"] = self.cfg.lr_attention_v
            elif "o_proj" in name:
                group_meta["optimizer_type"] = "muon"
                group_meta["lr"] = self.cfg.lr_attention_o
            elif any(x in name for x in ["gate_proj", "up_proj", "down_proj"]):
                group_meta["optimizer_type"] = "muon"
                group_meta["lr"] = self.cfg.lr_experts * self.cfg.scale_experts
            elif "mamba" in name:
                group_meta["optimizer_type"] = "muon"
                group_meta["lr"] = self.cfg.lr_mamba * self.cfg.scale_mamba
            else:
                group_meta["optimizer_type"] = "adamw"
                group_meta["lr"] = 1e-4

            group_meta["base_lr"] = group_meta["lr"]
            groups.append(group_meta)

        return groups


# =====================================================================
# 2. Host-Offloaded Caching Verification Runtime
# =====================================================================
class PretrainingOrchestrator:
    """
    Main orchestration execution pipeline wrapper. Manages sequential local-to-global
    gradient clipping bounds alongside isolated system memory shadow copies.
    """
    def __init__(self, model: nn.Module, optimizer: HybridOptimizerOrchestrator, train_cfg: TrainingConfig):
        self.model = model
        self.optimizer = optimizer
        self.cfg = train_cfg

        self.ema_state: Dict[str, torch.Tensor] = {}
        self.init_ema_buffers()

    def init_ema_buffers(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.ema_state[name] = param.data.cpu().clone().float()

    @torch.no_grad()
    def update_ema_weights(self):
        decay = self.cfg.ema_decay
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                device_data = param.data.cpu().float()
                self.ema_state[name].mul_(decay).add_(device_data, alpha=1.0 - decay)

    def run_step(self, batch_inputs: torch.Tensor, target_labels: torch.Tensor, is_hessian_warmup: bool = False) -> Dict[str, float]:
        self.optimizer.zero_grad()

        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
            # Direct compatibility path to capture MiniMixtral loss metrics
            logits, total_lb_loss, total_z_loss = self.model(batch_inputs)
            total_lb_loss = total_lb_loss.mean()
            total_z_loss = total_z_loss.mean()
            vocab_loss = F.cross_entropy(logits.view(-1, logits.size(-1)), target_labels.view(-1))
            total_loss = vocab_loss + (self.cfg.weight_load_balancing * total_lb_loss) + (self.cfg.weight_router_z * total_z_loss)
        total_loss.backward()

        # Guard exit criteria for initial warmups
        if is_hessian_warmup:
            self.optimizer.update_sophia_hessian()
            self.optimizer.zero_grad()
            return {"loss": float(total_loss.item())}
        self.optimizer.update_sophia_hessian()
        # Step 1: Isolate and clip router components at 0.1 locally
        router_params = [p for n, p in self.model.named_parameters() if "block_sparse_moe.gate" in n and p.grad is not None]
        if router_params:
            torch.nn.utils.clip_grad_norm_(router_params, max_norm=self.cfg.clip_router)

        # Step 2: Isolate and clip expert components at 0.5 locally
        expert_params = [p for n, p in self.model.named_parameters() if any(x in n for x in ["gate_proj", "up_proj", "down_proj"]) and p.grad is not None]
        if expert_params:
            torch.nn.utils.clip_grad_norm_(expert_params, max_norm=self.cfg.clip_expert)

        # Step 3: Global tracking normalization at 1.0
        global_grad_norm = torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=self.cfg.clip_global)


        self.optimizer.step()
        self.update_ema_weights()

        return {
            "loss": float(total_loss.item()),
            "vocab_loss": float(vocab_loss.item()),
            "lb_loss": float(total_lb_loss.item()),
            "z_loss": float(total_z_loss.item()),
            "global_norm": float(global_grad_norm.item())
        }


# =====================================================================
# 3. Execution Mock Dataset Utility
# =====================================================================
class SyntheticCausalTokensDataset(Dataset):
    def __init__(self, vocab_size: int, seq_len: int, num_samples: int = 1200):
        self.vocab_size = vocab_size
        self.seq_len = seq_len
        self.num_samples = num_samples

    def __len__(self) -> int:
        return self.num_samples

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        input_ids = torch.randint(0, self.vocab_size - 1, (self.seq_len,))
        labels = torch.roll(input_ids, shifts=-1, dims=0)
        labels[-1] = 0
        return input_ids, labels


import torch
import os

def save_training_checkpoint(model, optimizer, scheduler, current_step, filepath):
    """
    Serializes core training state to disk.
    Strips DataParallel wrappers to ensure hardware-agnostic checkpoints.
    """
    # 1. Safely unwrap the model to avoid 'module.' prefix pollution
    base_model = model.module if hasattr(model, "module") else model

    # 2. Map parameters to their string architecture names 
    param_to_name = {}
    for group in optimizer.param_groups:
        p = group["params"][0]
        param_to_name[p] = group["name"]

    # 3. Extract state buffers from Muon driver
    muon_saved_state = {}
    for p, state in optimizer.muon_driver.state.items():
        if p in param_to_name:
            p_name = param_to_name[p]
            muon_saved_state[p_name] = {k: v.cpu().clone() if isinstance(v, torch.Tensor) else v for k, v in state.items()}

    # 4. Extract state buffers from Sophia driver
    sophia_saved_state = {}
    for p, state in optimizer.sophia_driver.state.items():
        if p in param_to_name:
            p_name = param_to_name[p]
            sophia_saved_state[p_name] = {k: v.cpu().clone() if isinstance(v, torch.Tensor) else v for k, v in state.items()}

    # 5. Construct unified state payload using the UNWRAPPED model
    checkpoint_payload = {
        "step": current_step,
        "model_state": base_model.state_dict(), 
        "scheduler_state": scheduler.state_dict(),
        "muon_state": muon_saved_state,
        "sophia_state": sophia_saved_state,
        "adamw_state": optimizer.adamw_driver.state_dict(),
        "rng_state": torch.get_rng_state(),
        "cuda_rng_state": torch.cuda.get_rng_state() if torch.cuda.is_available() else None
    }

    # Secure storage write out
    os.makedirs(os.path.dirname(filepath) if os.path.dirname(filepath) else '.', exist_ok=True)
    torch.save(checkpoint_payload, filepath)
    print(f"[CHECKPOINT SAVED] Core training state synchronized to {filepath} at step {current_step}")

def load_training_checkpoint(filepath, model, optimizer, scheduler, device):
    """
    Restores complete model weights and preconditioned optimizer buffers.
    Includes strict prefix sanitization for legacy checkpoints.
    """
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Target core checkpoint file missing: {filepath}")

    print(f"[CHECKPOINT RESTORED] Resuming core elements from: {filepath}")
    checkpoint_payload = torch.load(filepath, map_location=device)

    # 1. Sanitize the incoming state dictionary (Removes legacy 'module.' prefixes)
    clean_state_dict = {}
    for key, value in checkpoint_payload["model_state"].items():
        clean_key = key[7:] if key.startswith("module.") else key
        clean_state_dict[clean_key] = value

    # 2. Safely unwrap the target model and load the clean weights
    base_model = model.module if hasattr(model, "module") else model
    base_model.load_state_dict(clean_state_dict)

    # 3. Restore Scheduler Progress
    scheduler.load_state_dict(checkpoint_payload["scheduler_state"])

    # 4. Build map of string names back to the current active parameter instances
    name_to_param = {}
    for group in optimizer.param_groups:
        p = group["params"][0]
        name_to_param[group["name"]] = p

    # 5. Restore Muon Driver internal history
    optimizer.muon_driver.state.clear()
    for p_name, state in checkpoint_payload["muon_state"].items():
        if p_name in name_to_param:
            p = name_to_param[p_name]
            optimizer.muon_driver.state[p] = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in state.items()}

    # 6. Restore Sophia Driver curvature history
    optimizer.sophia_driver.state.clear()
    for p_name, state in checkpoint_payload["sophia_state"].items():
        if p_name in name_to_param:
            p = name_to_param[p_name]
            optimizer.sophia_driver.state[p] = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in state.items()}

    # 7. Restore Native AdamW internal tracking weights
    optimizer.adamw_driver.load_state_dict(checkpoint_payload["adamw_state"])

    # 8. Re-apply stochastic randomness states
    torch.set_rng_state(checkpoint_payload["rng_state"].cpu())
    if checkpoint_payload["cuda_rng_state"] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state(checkpoint_payload["cuda_rng_state"].cpu())

    print(f"  -> Model weights and multi-driver optimizer matrices successfully mapped.")
    print(f"  -> Resuming computation at Step {checkpoint_payload['step']}.\n")

    return checkpoint_payload["step"]



In [5]:
import os
from kaggle_secrets import UserSecretsClient

# Retrieve the token from Kaggle Secrets
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

# Set it as an environment variable for Hugging Face libraries
os.environ["HF_TOKEN"] = hf_token

In [6]:
def get_disk_usage_mb(directory: str) -> float:
    """Returns total size of all .bin files in directory in MB."""
    total = sum(
        os.path.getsize(f)
        for f in _list_bin_files(directory)
        if os.path.exists(f)
    )
    return total / 1e6

def _list_bin_files(cache_dir: str, suffix: str = ".bin") -> List[str]:
    return glob.glob(os.path.join(cache_dir, f"*{suffix}"))


def _wait_for_file(path: str, timeout: float = 300.0, poll: float = 0.1) -> bool:
    """Block until path exists or timeout expires."""
    deadline = time.time() + timeout
    while not os.path.exists(path):
        if time.time() > deadline:
            return False
        time.sleep(poll)
    time.sleep(0.05)  # Let rename fully propagate on network filesystems.
    return True


In [ ]:
def main(total_steps = 10000):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.manual_seed(42)

    opt_cfg = OptimizerConfig()
    train_cfg = TrainingConfig(total_steps=total_steps,
                            train_resume = False) # Set to your desired testing or production step limit

    args = ModelArgs()
    args.vocab_size = 50257
    args.dim = 512
    args.n_layers = 32
    args.use_gradient_checkpointing = True
    print(f"we are running the model with {args.vocab_size} vocab size and using use_gradient_checkpointing as {args.use_gradient_checkpointing} and training on {total_steps} steps")
    model = MiniMixtral(args).to(device)
    model =  nn.DataParallel(model)

    # 1. Instantiate parameter routing and optimizers
    router = TiedAwareParameterRouter(model, opt_cfg)
    param_groups = router.build_groups()
    optimizer = HybridOptimizerOrchestrator(param_groups, opt_cfg)
    scheduler = WarmupPlateauCosineScheduler(optimizer, total_steps=train_cfg.total_steps)
    orchestrator = PretrainingOrchestrator(model, optimizer, train_cfg)

    checkpoint_dir = "/kaggle/working/checkpoints"
    dataset_ckpt_path = os.path.join(checkpoint_dir, "dataset_state.json")
    training_ckpt_path = os.path.join(checkpoint_dir, "training_state.pt")
    config = PipelineConfig(
        context_len      = 1024,
        chunk_sequences  = 4096,   # 4096 sequences per chunk → tiny files for demo
        batch_size       = 32,
        max_chunks_ahead = 12,   # producer pauses after 12 unconsumed chunks
    )
    if train_cfg.train_resume:
        re_step = load_training_checkpoint(training_ckpt_path, model, optimizer, scheduler, device)
        config = load_dataset_checkpoint(dataset_ckpt_path)
    # 2. Spin up the asynchronous text-to-binary producer pipeline
    
    text_provider = mock_raw_text_stream()
    producer = AsyncBinaryProducer(text_provider, config)
    producer.start()

    print("Waiting for background producer to initialize files on disk...")
    time.sleep(3.0) # Gives the producer a safe window to write chunk_0.bin

    # 3. Create the real JIT dataset and iterator streams
    batch_size = 8
    dataset = StreamingBinaryDataset(config, max_expected_chunks=1000000)
    dataloader = DataLoader(dataset, batch_size=batch_size, num_workers=0)
    data_iter = iter(dataloader)

    # 4. FIXED ORDER: Execute Sophia step-0 contract pass using actual language data
    print("Executing production step-0 hessian initialization pass on real text data...")
    init_batch = next(data_iter)
    init_inputs = init_batch['input_ids'].to(device)
    init_targets = init_batch['labels'].to(device)

    orchestrator.run_step(init_inputs, init_targets, is_hessian_warmup=True)
    print("Sophia matrix targets locked on real text syntax. Optimization pass initialized.\n")

    try:
        for step, batch in enumerate(data_iter):
            if train_cfg.train_resume:
                step += re_step
            inputs, targets = batch['input_ids'].to(device), batch['labels'].to(device)
            # Clip gradients to a maximum norm of 1.0
            # Advance scheduler learning rates
            scheduler.step()
            # Run the active weight update pass
            metrics = orchestrator.run_step(inputs, targets)
            # Log progress tracking metrics
            if step % 100 == 0 or step == train_cfg.total_steps - 1:
                print(
                    f"Step: {step:04d} | Total Loss: {metrics['loss']:.4f} | "
                    f"Vocab: {metrics['vocab_loss']:.4f} | LB: {metrics['lb_loss']:.4f} | "
                    f"Z-Loss: {metrics['z_loss']:.4f} | Norm: {metrics['global_norm']:.3f}"
                )
            if step > 0 and step % 500 == 0:
                print(f"\nCheckpoint interval reached at step {step}. Synchronizing states...")
                # 2. Serialize weights, Hessians, AdamW moments, and RNG seeds to the binary file
                save_training_checkpoint(model, optimizer, scheduler, current_step=step, filepath=training_ckpt_path)
                save_dataset_checkpoint(                    
                    config=config,
                    producer=producer,
                    dataloader_step=step,
                    dataloader_batch_size=batch_size,
                    filepath=dataset_ckpt_path )
            step += 1 # Strict step increments
    finally:
        # Prevent thread lockups by ensuring background resources close safely on finish or crash
        producer.shutdown()
        producer.join()

    print("Training run finalized successfully.")
    print(f"\nDisk after crash: {get_disk_usage_mb(config.cache_dir):.2f} MB")
    print(f"Chunks remaining on disk: {len(_list_bin_files(config.cache_dir))}")

if __name__ == "__main__":
    main(total_steps = 70000)